# Decimal Scaling

---

## Pengertian

**Decimal Scaling** merupakan salah satu teknik normalisasi data yang dilakukan dengan cara **menggeser titik desimal** dari nilai-nilai pada suatu atribut. Metode ini memperkecil nilai dari suatu atribut numerik dengan cara membagi nilai data dengan **pangkat sepuluh**.

Tujuan utamanya adalah mengubah skala nilai data sehingga seluruh nilai tersebut berada dalam rentang antara **-1 hingga 1**. Teknik ini sangat berguna dalam tahap preprocessing data mining atau machine learning, terutama jika algoritma yang digunakan sensitif terhadap perbedaan skala antar variabel.

## Rumus

$$v' = \frac{v}{10^j}$$

**Keterangan:**
- $v'$ : nilai baru setelah dinormalisasi
- $v$ : nilai asli dari data
- $j$ : bilangan bulat terkecil sehingga nilai absolut maksimum dari hasil normalisasi **lebih kecil dari 1**, yaitu jumlah digit dari nilai absolut terbesar

## Cara Menentukan Nilai $j$

1. Cari nilai **absolut terbesar** pada atribut
2. Hitung **jumlah digit** dari nilai tersebut
3. Nilai tersebut adalah $j$

**Contoh:**  
Nilai terbesar = 4.000.000 → memiliki **7 digit** → $j = 7$ → $10^7 = 10.000.000$

## Kelebihan dan Kekurangan

| Kelebihan | Kekurangan |
|-----------|------------|
| Sangat sederhana dan cepat | Tidak selalu menghasilkan rentang [0, 1] |
| Tidak mengubah distribusi data | Nilai $j$ bergantung pada data — jika ada data baru, $j$ bisa berubah |
| Cocok untuk data berskala besar | Kurang tepat jika nilai data memiliki variasi digit yang besar |
| Mudah diimplementasikan | |

## Dataset

| No | IPK | PO        | JML |
|----|-----|-----------|-----|
| 1  | 2   | 2.000.000 | 2   |
| 2  | 3   | 3.000.000 | 3   |
| 3  | 4   | 2.000.000 | 2   |
| 4  | 2   | 2.000.000 | 3   |
| 5  | 3   | 3.000.000 | 2   |
| 6  | 4   | 4.000.000 | 3   |

## Contoh Perhitungan Manual — Kolom PO

**Langkah 1 — Temukan nilai absolut terbesar:**

Nilai PO = {2.000.000, 3.000.000, 2.000.000, 2.000.000, 3.000.000, 4.000.000}

Nilai terbesar = **4.000.000**

**Langkah 2 — Hitung jumlah digit:**

4.000.000 memiliki **7 digit** → $j = 7$

**Langkah 3 — Hitung $10^j$:**

$$10^7 = 10.000.000$$

**Langkah 4 — Hitung Decimal Scaling setiap nilai:**

$$v_1' = \frac{2.000.000}{10.000.000} = 0.2$$

$$v_2' = \frac{3.000.000}{10.000.000} = 0.3$$

$$v_3' = \frac{2.000.000}{10.000.000} = 0.2$$

$$v_4' = \frac{2.000.000}{10.000.000} = 0.2$$

$$v_5' = \frac{3.000.000}{10.000.000} = 0.3$$

$$v_6' = \frac{4.000.000}{10.000.000} = 0.4$$

Seluruh nilai hasil normalisasi berada dalam rentang 0 sampai 1 ✅

## Implementasi Python — Fungsi Manual

In [ ]:
import pandas as pd
import numpy as np

# Dataset
data = {
    'No':  [1, 2, 3, 4, 5, 6],
    'IPK': [2, 3, 4, 2, 3, 4],
    'PO':  [2000000, 3000000, 2000000, 2000000, 3000000, 4000000],
    'JML': [2, 3, 2, 3, 2, 3]
}
df = pd.DataFrame(data).set_index('No')

print("=== Data Sebelum Normalisasi ===")
print(df)

In [ ]:
# =========================================
# Fungsi Decimal Scaling Manual
# =========================================
def decimal_scaling(df, kolom):
    """
    Melakukan Decimal Scaling pada kolom yang dipilih.
    
    Parameters:
        df    : DataFrame
        kolom : list kolom yang akan dinormalisasi
    
    Returns:
        df_out  : DataFrame hasil normalisasi
        info_j  : dict berisi nilai j untuk setiap kolom
    """
    df_out = df.copy()
    info_j = {}
    
    for col in kolom:
        max_abs = df[col].abs().max()
        
        # Hitung jumlah digit
        j = int(np.floor(np.log10(max_abs))) + 1
        
        # Normalisasi
        df_out[col] = df[col] / (10**j)
        info_j[col] = j
        
        print(f"Kolom {col}: nilai terbesar = {max_abs}, j = {j}, pembagi = 10^{j} = {10**j}")
    
    return df_out, info_j

kolom_target = ['IPK', 'PO', 'JML']
df_dec, info_j = decimal_scaling(df, kolom_target)

print("\n=== Hasil Decimal Scaling (Fungsi Manual) ===")
print(df_dec.round(6))

In [ ]:
# Detail perhitungan per objek untuk kolom PO
print("=== Detail Perhitungan Decimal Scaling Kolom PO ===")
j_po   = info_j['PO']
div_po = 10**j_po
print(f"Nilai absolut terbesar : {df['PO'].abs().max():,}")
print(f"Jumlah digit (j)       : {j_po}")
print(f"Pembagi (10^j)         : {div_po:,}\n")

print(f"{'No':<5} {'PO (asli)':<15} {'Perhitungan':<30} {'Hasil'}")
print("-" * 65)
for idx, val in df['PO'].items():
    hasil = val / div_po
    print(f"{idx:<5} {val:<15,} {val:,} / {div_po:,} = {hasil:<15.4f}")

## Implementasi Python — Menggunakan sklearn

In [ ]:
from sklearn.preprocessing import MaxAbsScaler

# =========================================
# MaxAbsScaler dari sklearn
# =========================================
# MaxAbsScaler membagi dengan nilai absolut maksimum dari setiap fitur.
# Konsep ini serupa dengan Decimal Scaling, namun menggunakan nilai absolut
# maksimum secara langsung (bukan 10^j).

mas = MaxAbsScaler()

df_sklearn = df.copy()
df_sklearn[kolom_target] = mas.fit_transform(df[kolom_target])

print("=== Hasil dengan sklearn MaxAbsScaler ===")
print("(MaxAbsScaler membagi tiap nilai dengan nilai absolut maksimumnya)")
print(df_sklearn.round(6))

print("\nNilai absolut maks per kolom (dari scaler):", mas.max_abs_)

In [ ]:
# =========================================
# Perbandingan Decimal Scaling vs MaxAbsScaler
# =========================================
print("=== Perbandingan: Decimal Scaling vs MaxAbsScaler (Kolom PO) ===")
print(f"{'No':<5} {'PO Asli':<15} {'Decimal Scaling':<20} {'MaxAbsScaler'}")
print("-" * 55)
for idx in df.index:
    asli = df.loc[idx, 'PO']
    dec  = df_dec.loc[idx, 'PO']
    mas_ = df_sklearn.loc[idx, 'PO']
    print(f"{idx:<5} {asli:<15,} {dec:<20.4f} {mas_:.4f}")

print("\nCatatan:")
print("  Decimal Scaling  : membagi dengan 10^j = 10^7 = 10.000.000")
print("  MaxAbsScaler     : membagi dengan nilai maks absolut = 4.000.000")
print("  → Keduanya serupa secara konsep namun nilai pembaginya berbeda.")

## Kesimpulan

- Decimal Scaling menormalisasi data dengan membagi setiap nilai dengan **$10^j$**, di mana $j$ adalah jumlah digit dari nilai absolut terbesar
- Hasil normalisasi berada dalam rentang **(−1, 1)** namun tidak selalu sama dengan [0, 1]
- Metode ini sangat **sederhana** dan cepat, cocok untuk data berskala besar
- Tidak mengubah distribusi data secara signifikan
- sklearn tidak memiliki implementasi Decimal Scaling secara langsung, namun **MaxAbsScaler** menggunakan konsep yang serupa